In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    radius=1000,
    fletcher=False,
    c1=1e-4,
    tau=0.5,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=1000
    # batch_size=None
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4825890362262726
epoch:  1, loss: 0.15333524346351624
epoch:  2, loss: 0.09843302518129349
epoch:  3, loss: 0.06555824726819992
epoch:  4, loss: 0.05880717933177948
epoch:  5, loss: 0.012834353372454643
epoch:  6, loss: 0.00940497312694788
epoch:  7, loss: 0.008172954432666302
epoch:  8, loss: 0.007482470478862524
epoch:  9, loss: 0.006508370395749807
epoch:  10, loss: 0.006064579356461763
epoch:  11, loss: 0.0057411822490394115
epoch:  12, loss: 0.005037493538111448
epoch:  13, loss: 0.004501002375036478
epoch:  14, loss: 0.00421342346817255
epoch:  15, loss: 0.0035113783087581396
epoch:  16, loss: 0.003099365858361125
epoch:  17, loss: 0.002866661176085472
epoch:  18, loss: 0.002361742313951254
epoch:  19, loss: 0.0022508767433464527
epoch:  20, loss: 0.0021946877241134644
epoch:  21, loss: 0.002103238133713603
epoch:  22, loss: 0.0020729354582726955
epoch:  23, loss: 0.002051467774435878
epoch:  24, loss: 0.0019705449230968952
epoch:  25, loss: 0.00193935900460928

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9980359372117633
Test metrics:  R2 = 0.9947367373704454


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:44: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.3484644293785095
epoch:  1, loss: 0.24854259192943573
epoch:  2, loss: 0.23298503458499908
epoch:  3, loss: 0.09177333861589432
epoch:  4, loss: 0.07500278204679489
epoch:  5, loss: 0.030719319358468056
epoch:  6, loss: 0.012366634793579578
epoch:  7, loss: 0.010902647860348225
epoch:  8, loss: 0.009952504187822342
epoch:  9, loss: 0.009728560224175453
epoch:  10, loss: 0.009437395259737968
epoch:  11, loss: 0.009264660999178886
epoch:  12, loss: 0.009073100984096527
epoch:  13, loss: 0.009030773304402828
epoch:  14, loss: 0.007043807301670313
epoch:  15, loss: 0.00674273120239377
epoch:  16, loss: 0.006543781608343124
epoch:  17, loss: 0.006523266434669495
epoch:  18, loss: 0.0062537044286727905
epoch:  19, loss: 0.005761384963989258
epoch:  20, loss: 0.0056366268545389175
epoch:  21, loss: 0.00554857961833477
epoch:  22, loss: 0.005486007779836655
epoch:  23, loss: 0.005439411383122206
epoch:  24, loss: 0.005387471057474613
epoch:  25, loss: 0.005328032188117504
epoch:  26,

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9972611692728324
Test metrics:  R2 = 0.9961831215786295
